In [0]:
%sql
CREATE OR REPLACE TABLE small_projects.finops.gold_usage_cost AS
SELECT
  u.*,
  r.resource_name,
  r.resource_type,
  r.environment,
  r.dbu_per_hour,
  r.vm_cost_per_hour_usd,
  r.auto_termination_minutes,
  p.price_per_dbu_usd,
  CAST(date_trunc('MONTH', u.usage_date) AS DATE)                         AS usage_month,
  u.dbus * p.price_per_dbu_usd                                            AS dbu_cost,
  u.uptime_hours * r.vm_cost_per_hour_usd                                 AS vm_cost,
  u.dbus * p.price_per_dbu_usd + u.uptime_hours * r.vm_cost_per_hour_usd  AS total_cost,
  r.dbu_per_hour * p.price_per_dbu_usd + r.vm_cost_per_hour_usd           AS hourly_cost,
  u.uptime_hours - u.activate_hours                                       AS idle_hours,
  (u.uptime_hours - u.activate_hours)
    * (r.dbu_per_hour * p.price_per_dbu_usd + r.vm_cost_per_hour_usd)     AS idle_cost,
  CASE WHEN r.auto_termination_minutes = 0 THEN
         (u.uptime_hours - CASE WHEN u.activate_hours > 0 THEN u.activate_hours + 0.5 ELSE 0 END)
         * (r.dbu_per_hour * p.price_per_dbu_usd + r.vm_cost_per_hour_usd)
       ELSE 0 END                                                         AS autoterm_saving,
  CASE WHEN u.sku_name = 'ALL_PURPOSE_COMPUTE'
        AND u.job_name IS NOT NULL AND u.job_name <> ''
       THEN u.dbus * (0.55 - 0.30) ELSE 0 END                             AS jobcompute_saving
FROM small_projects.finops.silver_billing_usage u
JOIN small_projects.finops.silver_resources  r ON u.resource_id = r.resource_id
JOIN small_projects.finops.silver_sku_prices p ON u.sku_name    = p.sku_name;

In [0]:
%sql
SELECT
  COUNT(*)                                                               AS row_count,
  ROUND(SUM(total_cost))                                                 AS total_cost,
  ROUND(SUM(idle_cost))                                                  AS idle_cost,
  ROUND(SUM(autoterm_saving))                                            AS autoterm_saving,
  ROUND(SUM(jobcompute_saving))                                          AS jobcompute_saving,
  ROUND(SUM(CASE WHEN tag_team = 'untagged' THEN total_cost ELSE 0 END)) AS untagged_cost
FROM small_projects.finops.gold_usage_cost;

In [0]:
%sql
SELECT usage_month, ROUND(SUM(total_cost)) AS monthly_cost
FROM small_projects.finops.gold_usage_cost
GROUP BY usage_month
ORDER BY usage_month;

In [0]:
%sql
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN activate_hours COMMENT 'Hours the resource was busy running workloads (active time)';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN uptime_hours COMMENT 'Hours the resource was running and billed (busy + idle)';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN total_cost COMMENT 'Cost in USD for this resource on this day = DBU cost + Azure VM cost. Sum it for totals.';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN hourly_cost COMMENT 'Cost in USD for each hour the resource is running, whether busy or idle';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN autoterm_saving COMMENT 'Potential (not realized) USD saving if 30-minute auto-termination or auto-stop were turned on';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN jobcompute_saving COMMENT 'Potential (not realized) USD saving if scheduled jobs ran on job compute instead of all-purpose compute';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN auto_termination_minutes COMMENT 'Auto-termination or auto-stop setting in minutes; 0 means turned off; empty for job clusters';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN tag_team COMMENT 'Team that owns the resource; untagged means no owner';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN resource_type COMMENT 'All-purpose cluster, Job cluster or SQL warehouse';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN environment COMMENT 'prod or dev';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN job_name COMMENT 'Scheduled job that used the resource; empty for interactive or SQL warehouse use';
ALTER TABLE small_projects.finops.gold_usage_cost ALTER COLUMN sku_name COMMENT 'Compute type used for pricing: ALL_PURPOSE_COMPUTE, JOBS_COMPUTE or SQL_PRO_COMPUTE';